In [ ]:
# 安装依赖
!pip install -q transformers torch torchaudio soundfile librosa

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 下载 LibriSpeech 数据集里的一段语音
import urllib.request
url = "https://github.com/openai/whisper/raw/main/tests/jfk.flac"
urllib.request.urlretrieve(url, "jfk.flac")

# 播放
from IPython.display import Audio
Audio("jfk.flac")

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torchaudio

# 加载小模型（39MB，跑得快）
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en").to("cuda")

# 加载音频
waveform, sample_rate = torchaudio.load("jfk.flac")
# Whisper 要求 16kHz
if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(sample_rate, 16000)
    waveform = resampler(waveform)
    sample_rate = 16000

# 预处理
input_features = processor(
    waveform.squeeze().numpy(),
    sampling_rate=16000,
    return_tensors="pt"
).input_features.to("cuda")

# 识别
predicted_ids = model.generate(input_features)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
print("识别结果:", transcription[0])